# K 近邻算法
- 监督学习——即可以做分类，又可以做回归
- 过程：
> 1. 选定 K 值和距离度量方式
> 2. 计算待预测样本与训练集中每个样本的距离
> 3. 取距离最近的 K 个样本
> 4. 分类:K 个邻居中多数票的类别即为预测结果;回归:取 K 个邻居目标值的平均值(或加权平均)

In [29]:
from collections.abc import generator

import torch
import torch.nn.functional as F

x = torch.randn(2, 3)
y = torch.randn(3, 3)

### 距离度量方式
- 欧氏距离
- $distance = \sqrt{\sum{(a_i - b_i)^2}}$

In [32]:
def euclidean(x, y):
    d2 = (x**2).sum(1, keepdim=True) + (y**2).sum(1) - 2 * x @ y.T
    return d2.clamp_min(0).sqrt()      # clamp 防止浮点误差出现负数
def euclidean2(x, y):
    d1 = x[:, None, :] - y[None, :, :]
    d2 = d1.pow(2).sum(-1)
    return d2.clamp_min(0).sqrt()      # clamp 防止浮点误差出现负数
print(euclidean2(x, y))
print(euclidean(x, y))
torch.cdist(x, y, p=2)                 # 库写法

tensor([[2.5251, 2.2589, 1.6283],
        [1.6565, 1.5842, 2.7313]])
tensor([[2.5251, 2.2589, 1.6283],
        [1.6565, 1.5842, 2.7313]])


tensor([[2.5251, 2.2589, 1.6283],
        [1.6565, 1.5842, 2.7313]])

In [40]:
d1 = x[:, None, :] - y[None, :, :]
d2 = d1.pow(2).sum(-1)
print(d2)
print(d2.clamp_min(0))
d2.clamp_min(0).sqrt()

tensor([[6.3763, 5.1027, 2.6514],
        [2.7441, 2.5097, 7.4602]])
tensor([[6.3763, 5.1027, 2.6514],
        [2.7441, 2.5097, 7.4602]])


tensor([[2.5251, 2.2589, 1.6283],
        [1.6565, 1.5842, 2.7313]])

- 曼哈顿距离
- $distance = \sum |xᵢ − yᵢ|$

In [8]:
def manhattan(x, y):
    return (x[:, None, :] - y[None, :, :]).abs().sum(-1)
print(manhattan(x, y))
torch.cdist(x, y, p=1)

tensor([[3., 3., 3.],
        [3., 3., 3.]])


tensor([[3., 3., 3.],
        [3., 3., 3.]])

- 余弦距离
- $distance = 1 − \frac{(x·y)}{‖x‖‖y‖} = 1 − \frac{\sum{x_i·y_i}}{\sum{x_i^2}\times \sum{y_i^2}}$

In [9]:
def cosine_distance(x, y):
    return 1 - F.normalize(x, dim=1) @ F.normalize(y, dim=1).T

print(cosine_distance(x, y))
1 - F.cosine_similarity(x[:, None, :], y[None, :, :], dim=-1)

tensor([[1., 1., 1.],
        [1., 1., 1.]])


tensor([[1., 1., 1.],
        [1., 1., 1.]])

- 一些涉及到的torch计算方式

In [24]:
x = torch.tensor([[1,2,3],
                   [1,1,1]])
y = torch.tensor([[0,0,1],
                   [1,0,0],
                   [0,1,0]])
print(x[:, None, :] - y[None, :, :])

tensor([[[1, 2, 2],
         [0, 2, 3],
         [1, 1, 3]],

        [[1, 1, 0],
         [0, 1, 1],
         [1, 0, 1]]])


In [20]:
print(x,x.shape) # 2 * 3
print(x[:, None, :],x[:, None, :].shape)

tensor([[1, 2, 3],
        [1, 1, 1]]) torch.Size([2, 3])
tensor([[[1, 2, 3]],

        [[1, 1, 1]]]) torch.Size([2, 1, 3])


In [21]:
print(y,y.shape) # 3 * 3
print(y[None,:, :],y[None,:, :].shape)

tensor([[0, 0, 1],
        [1, 0, 0],
        [0, 1, 0]]) torch.Size([3, 3])
tensor([[[0, 0, 1],
         [1, 0, 0],
         [0, 1, 0]]]) torch.Size([1, 3, 3])


In [33]:
x

tensor([[-0.2561,  0.4918, -0.6113],
        [ 0.2319,  1.1050,  0.3903]])

In [36]:
x.sum(dim=1, keepdim=True)

tensor([[-0.3757],
        [ 1.7272]])

In [37]:
x.sum(dim=1)

tensor([-0.3757,  1.7272])

## 选择邻居

In [41]:
import torch
import torch.nn.functional as F


def knn_predict(
    train_x: torch.Tensor,
    train_y: torch.Tensor,
    query_x: torch.Tensor,
    k: int,
) -> torch.Tensor:
    d2 = (query_x[:, None, :] - train_x[None, :, :]).pow(2).sum(-1)   # (Q, N)
    idx = torch.argsort(d2, dim=1, stable=True)[:, :k]                # (Q, k)
    neighbors = train_y[idx]                                          # (Q, k)
    num_classes = int(train_y.max()) + 1
    votes = F.one_hot(neighbors, num_classes).sum(1)                  # (Q, C)
    return votes.argmax(1).to(torch.int64)

train_x = torch.tensor([[0,0],[1,1],[5,5],[6,6]])
train_y = torch.tensor([0,0,1,1])
query_x = torch.tensor([[0.5,0.5],[5.5,5.5]])
k = 3

print(knn_predict(train_x, train_y, query_x, k))

tensor([0, 1])


## 一个示例：KNN识别手写数字

- 从 `scikit-learn` 里面导入数据，`load_digits()`

In [65]:
import torch
from sklearn.datasets import load_digits

torch.set_num_threads(1)
digits = load_digits()
# images[i] 是第 i 张 8×8 图片，y[i] 是它的标签，即图片上写的数字。
images = torch.tensor(digits.images, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
print(f"图片 Tensor 的形状：{tuple(images.shape)},共{images.shape[0]}张图片，每张图片{images.shape[1]}*{images.shape[2]}")

图片 Tensor 的形状：(1797, 8, 8),共1797张图片，每张图片8*8


- 将图片展开成二维并归一化(`min-max`归一化)
- 因为KNN只接受二维矩阵输入

In [89]:
x = images.reshape(-1, 64)
x = x / x.max().item()

- 划分训练集，测试集，验证集

In [90]:
# 先打乱样本编号，再分成互不重叠的三份。随机种子固定为 42，每次运行得到同样的划分。
generator = torch.Generator().manual_seed(42)
order = torch.randperm(len(x), generator=generator)

train_ids = order[:1078]
val_ids = order[1078:1437]
test_ids = order[1437:]

x_train, y_train = x[train_ids], y[train_ids]
x_val, y_val = x[val_ids], y[val_ids]
x_test, y_test = x[test_ids], y[test_ids]


- 找到邻居

In [97]:
# 待识别图片在验证集中的下标，可以改成 0～358 之间的整数。
query_index = 6
# 参与投票的邻居数量。
k = 3
# 待识别图片的 64 个特征。
query = x_val[query_index]

# x_train 形状为 (1078, 64)，query 形状为 (64,)，广播后每张训练图片都减去同一个 query。
# sum(dim=1) 按行累加 64 个平方差，squared_distances[i] 就是 query 与第 i 张训练图片的距离平方。
# 只比较远近，不必开平方。
squared_distances = ((x_train - query) ** 2).sum(dim=1)
# 按距离从近到远排出训练集下标（等距时保持原顺序），取前 k 个就是最近邻。
neighbor_indices = squared_distances.argsort(stable=True)[:k]
# 最近的 k 个邻居的标签。
neighbor_labels = y_train[neighbor_indices]

# 开平方得到邻居的实际距离，用于打印和作图。
distances = squared_distances[neighbor_indices].sqrt()
print("邻居标签：", neighbor_labels.tolist())
print("距离：", [round(d, 3) for d in distances.tolist()])

邻居标签： [2, 2, 2]
距离： [0.912, 0.982, 1.168]


- 统计得票

In [98]:
# 统计 0～9 的票数，votes[i] 就是数字 i 得到的票数。
votes = torch.bincount(neighbor_labels, minlength=10)
# 票数最多的位置就是预测的数字。
prediction = votes.argmax().item()
print("0～9 的票数：", votes.tolist())
print(f"预测：{prediction}，实际：{y_val[query_index].item()}")

0～9 的票数： [0, 0, 3, 0, 0, 0, 0, 0, 0, 0]
预测：2，实际：2


- 进行封装

In [107]:
def predict_one(_query, _k):
    _distances = (x_train - _query).pow(2).sum(dim=1)
    _nearest_neighbors = _distances.argsort(stable=True)[:_k]
    _votes = torch.bincount(y_train[_nearest_neighbors], minlength=10)
    return _votes.argmax().item()

k = 5
val_predictions = torch.tensor([predict_one(query, k) for query in x_val])
wrong_indices = torch.where(val_predictions != y_val)[0]
print(f"k={k}：答对 {len(x_val) - len(wrong_indices)}/{len(x_val)} 张")
print("误判的验证集下标：", wrong_indices.tolist())

k=5：答对 355/359 张
误判的验证集下标： [41, 97, 162, 224]


- 决定k的取值

In [109]:
# 要比较的 k 值。
candidates = [1, 3, 5, 7, 9, 15, 31]
# correct_counts[i] 是 k=candidates[i] 时验证集答对的图片数。
correct_counts = []
# 每个候选 k 都在同一个验证集上逐张预测，统计答对的数量。
for candidate in candidates:
    predictions = torch.tensor([predict_one(query, candidate) for query in x_val])
    correct = (predictions == y_val).sum().item()
    correct_counts.append(correct)
    print(f"k={candidate:2d}：{correct}/{len(x_val)} = {correct / len(x_val):.2%}")

# 按验证集答对数量选 k，数量相同时取列表中靠前的值。
best_index = correct_counts.index(max(correct_counts))
best_k = candidates[best_index]
print("选定 k：", best_k)

k= 1：354/359 = 98.61%
k= 3：354/359 = 98.61%
k= 5：355/359 = 98.89%
k= 7：354/359 = 98.61%
k= 9：353/359 = 98.33%
k=15：346/359 = 96.38%
k=31：334/359 = 93.04%
选定 k： 5


In [110]:
# 使用验证集选出的 k，在测试集上评估。
test_predictions = torch.tensor([predict_one(query, best_k) for query in x_test])
correct = (test_predictions == y_test).sum().item()
print(f"测试集，k={best_k}：{correct}/{len(x_test)} = {correct / len(x_test):.2%}")

测试集，k=5：353/360 = 98.06%
